# Phase 4: Trajectory Extraction Exploration

This notebook explores converting Phase 3 `tracks.csv` outputs into ordered 2D ground-plane trajectories:
- Comparing `bottom_center` `(0.5*(x1+x2), y2)` vs `centroid` `(0.5*(x1+x2), 0.5*(y1+y2))` anchors.
- Evaluating causal rolling-mean smoothing (`window=1, 3, 5`) on coordinate jitter and path length.
- Exporting `trajectories.csv`, `track_summary.csv`, and `trajectory_map.png`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while (
    not (PROJECT_ROOT / "pyproject.toml").exists()
    and PROJECT_ROOT != PROJECT_ROOT.parent
):
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "cv_projects" / "04_trajectory_extraction" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from compare_smoothing import compare_smoothing
from extract_trajectories import (
    compute_trajectory_summaries,
    read_tracks_csv,
    smooth_trajectories,
    tracks_to_trajectories,
)

In [ ]:
tracks_csv = (
    PROJECT_ROOT
    / "cv_projects"
    / "03_person_tracking"
    / "outputs"
    / "people_detection_bytetrack"
    / "tracks.csv"
)
if tracks_csv.exists():
    frame_tracks = read_tracks_csv(tracks_csv)
    raw_bc = tracks_to_trajectories(frame_tracks, fps=12.0, anchor="bottom_center")
    smooth_bc = smooth_trajectories(raw_bc, window=3)
    summaries = compute_trajectory_summaries(smooth_bc, fps=12.0)
    for s in summaries:
        print(s.to_row())
    cmp_report = compare_smoothing(
        tracks_csv,
        PROJECT_ROOT
        / "cv_projects"
        / "04_trajectory_extraction"
        / "outputs"
        / "notebook_cmp",
        fps=12.0,
    )
    print(json.dumps(cmp_report, indent=2))